In [1]:
import logging
import os
from pathlib import Path
from dotenv import load_dotenv
from zerobus.sdk.sync import ZerobusSdk
from zerobus.sdk.shared import RecordType, StreamConfigurationOptions, TableProperties

# Credentials are loaded from .env at the repo root (not committed to git).
# Copy .env.example → .env at the repo root and fill in your values.
_repo_root = Path(__file__).parent.parent if "__file__" in dir() else Path.cwd().parent
load_dotenv(_repo_root / ".env")

SERVER_ENDPOINT          = os.environ["SERVER_ENDPOINT"]
DATABRICKS_WORKSPACE_URL = os.environ["DATABRICKS_WORKSPACE_URL"]
TABLE_NAME               = os.environ["TABLE_NAME"]
CLIENT_ID                = os.environ["CLIENT_ID"]
CLIENT_SECRET            = os.environ["CLIENT_SECRET"]

In [5]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {'.'.join(TABLE_NAME.split('.')[:2])}")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_NAME} (
  device_name STRING,
  temp        INT,
  humidity    INT
)
""")
spark.sql(f"GRANT MODIFY, SELECT ON TABLE {TABLE_NAME} TO `{CLIENT_ID}`")

""


In [6]:
sdk = ZerobusSdk(
    SERVER_ENDPOINT,
    DATABRICKS_WORKSPACE_URL
)

table_properties = TableProperties(TABLE_NAME)
options = StreamConfigurationOptions(record_type=RecordType.JSON)
stream = sdk.create_stream(CLIENT_ID, CLIENT_SECRET, table_properties, options)

try:
    for i in range(1000):
        record_dict = {
            "device_name": f"sensor-{i}",
            "temp": 20 + i % 15,
            "humidity": 50 + i % 40
        }
        offset = stream.ingest_record_offset(record_dict)

        # Optional: Wait for durability confirmation
        stream.wait_for_offset(offset)
finally:
    stream.close()

2026-03-18T16:37:01.534488Z ERROR databricks_zerobus_ingest_sdk: Stream initialization failed with error: Specified UC token is in invalid format: Client error (401): {"error":"invalid_authorization_details","request_id":"9f93c826-2fb6-4659-9956-9dd624ed0c5f","error_description":"User is not authorized to the requested authorizations"}.


ZerobusException: Specified UC token is in invalid format: Client error (401): {"error":"invalid_authorization_details","request_id":"9f93c826-2fb6-4659-9956-9dd624ed0c5f","error_description":"User is not authorized to the requested authorizations"}.